# 50 - Build Gold Service

Builds a Power BI-ready service-management star schema with conformed dimensions and materialized fact tables.

In [17]:
# Attach the schema-enabled Lakehouse containing the Bronze shortcuts and
# make it the default Lakehouse before running this notebook.
BRONZE_SCHEMA = "bronze"
SILVER_SCHEMA = "silver"
GOLD_SCHEMA = "gold"
METADATA_SCHEMA = "metadata"
COMPANY_KEY = "NAC"  # Change when processing another GP company database.

def q(identifier: str) -> str:
    return f"`{identifier.replace('`', '``')}`"

def sql_string(value: str) -> str:
    return "'" + value.replace("'", "''") + "'"

def table_exists(schema_name: str, table_name: str) -> bool:
    try:
        return spark.catalog.tableExists(f"{schema_name}.{table_name}")
    except Exception:
        return False

def require_tables(table_names: list[str], schema_name: str = BRONZE_SCHEMA) -> None:
    missing = [name for name in table_names if not table_exists(schema_name, name)]
    if missing:
        raise RuntimeError(
            f"Missing tables in {schema_name}: {', '.join(missing)}. "
            "Confirm the OneLake shortcuts and exact table names."
        )

company = sql_string(COMPANY_KEY)
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {q(SILVER_SCHEMA)}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {q(GOLD_SCHEMA)}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {q(METADATA_SCHEMA)}")
print(f"Company: {COMPANY_KEY}; Bronze schema: {BRONZE_SCHEMA}")

StatementMeta(, dffa1368-8c23-41d3-af84-999f7974401e, 19, Finished, Available, Finished, False)

Company: NAC; Bronze schema: bronze


In [18]:
require_tables(["customer", "location", "employee", "technician", "service_call",
                "service_call_hours", "service_appointment", "service_labor_actual",
                "service_cost", "service_contract", "contract_schedule",
                "contract_revenue_schedule"], SILVER_SCHEMA)

StatementMeta(, dffa1368-8c23-41d3-af84-999f7974401e, 20, Finished, Available, Finished, False)

## Conformed dimensions

Key `0` is reserved for unknown or late-arriving dimension members.

In [19]:
spark.sql(f"""
CREATE OR REPLACE TABLE {q(GOLD_SCHEMA)}.dim_customer
USING DELTA
AS
SELECT CAST(0 AS BIGINT) AS customer_key, {company} AS company_key,
       'UNKNOWN' AS customer_code, 'Unknown customer' AS customer_name,
       CAST(NULL AS STRING) AS customer_city, CAST(NULL AS STRING) AS customer_state,
       CAST(NULL AS STRING) AS customer_zip
UNION ALL
SELECT XXHASH64(company_key, customer_code), company_key, customer_code, customer_name,
       customer_city, customer_state, customer_zip
FROM {q(SILVER_SCHEMA)}.customer
""")

spark.sql(f"""
CREATE OR REPLACE TABLE {q(GOLD_SCHEMA)}.dim_location
USING DELTA
AS
SELECT CAST(0 AS BIGINT) AS location_key, {company} AS company_key,
       'UNKNOWN' AS customer_code, 'UNKNOWN' AS location_code,
       'Unknown location' AS location_name, CAST(NULL AS STRING) AS location_city,
       CAST(NULL AS STRING) AS location_state, CAST(NULL AS STRING) AS location_zip
UNION ALL
SELECT XXHASH64(company_key, customer_code, location_code), company_key,
       customer_code, location_code, location_name, location_city, location_state, location_zip
FROM {q(SILVER_SCHEMA)}.location
""")

spark.sql(f"""
CREATE OR REPLACE TABLE {q(GOLD_SCHEMA)}.dim_employee
USING DELTA
AS
SELECT CAST(0 AS BIGINT) AS employee_key, {company} AS company_key,
       CAST(NULL AS BIGINT) AS employee_code, 'Unknown employee' AS employee_name,
       CAST(NULL AS STRING) AS employee_job_code,
       CAST(NULL AS STRING) AS department_code, CAST(NULL AS STRING) AS department_name
UNION ALL
SELECT XXHASH64(company_key, CAST(employee_code AS STRING)), company_key, employee_code,
       CONCAT_WS(', ', employee_last_name, employee_first_name), employee_job_code,
       department_code, department_name
FROM {q(SILVER_SCHEMA)}.employee
""")

spark.sql(f"""
CREATE OR REPLACE TABLE {q(GOLD_SCHEMA)}.dim_technician
USING DELTA
AS
SELECT CAST(0 AS BIGINT) AS technician_key, {company} AS company_key,
       'UNKNOWN' AS technician_username, 'Unknown technician' AS technician_name,
       CAST(NULL AS STRING) AS technician_team, CAST(NULL AS BIGINT) AS employee_code
UNION ALL
SELECT XXHASH64(company_key, technician_username), company_key, technician_username,
       technician_name, technician_team, employee_code
FROM {q(SILVER_SCHEMA)}.technician
""")

spark.sql(f"""
CREATE OR REPLACE TABLE {q(GOLD_SCHEMA)}.dim_contract
USING DELTA
AS
WITH ranked_contracts AS (
    SELECT
        company_key,
        customer_code,
        location_code,
        contract_number,
        contract_sequence,
        contract_description,
        contract_status,
        start_date,
        end_date,
        annual_contract_value,
        ROW_NUMBER() OVER (
            PARTITION BY
                company_key,
                customer_code,
                location_code,
                contract_number,
                contract_sequence
            ORDER BY
                CASE
                    WHEN contract_status = 'Active' THEN 0
                    ELSE 1
                END,
                end_date DESC,
                start_date DESC
        ) AS row_rank
    FROM {q(SILVER_SCHEMA)}.service_contract
)

SELECT
    CAST(0 AS BIGINT) AS contract_key,
    {company} AS company_key,
    'UNKNOWN' AS contract_number,
    CAST(NULL AS BIGINT) AS contract_sequence,
    'Unknown contract' AS contract_description,
    CAST(NULL AS STRING) AS contract_status,
    CAST(NULL AS STRING) AS customer_code,
    CAST(NULL AS STRING) AS location_code,
    CAST(NULL AS DATE) AS start_date,
    CAST(NULL AS DATE) AS end_date,
    CAST(NULL AS DECIMAL(18,2)) AS annual_contract_value

UNION ALL

SELECT
    XXHASH64(
        company_key,
        customer_code,
        location_code,
        contract_number,
        contract_sequence
    ) AS contract_key,
    company_key,
    contract_number,
    contract_sequence,
    contract_description,
    contract_status,
    customer_code,
    location_code,
    start_date,
    end_date,
    annual_contract_value
FROM ranked_contracts
WHERE row_rank = 1
""")

spark.sql(f"""
CREATE OR REPLACE TABLE {q(GOLD_SCHEMA)}.dim_equipment
USING DELTA
AS
WITH equipment AS (
    SELECT company_key, equipment_code FROM {q(SILVER_SCHEMA)}.service_call_equipment
    UNION
    SELECT company_key, equipment_code FROM {q(SILVER_SCHEMA)}.contract_schedule_equipment
)
SELECT CAST(0 AS BIGINT) AS equipment_key, {company} AS company_key,
       'UNKNOWN' AS equipment_code
UNION ALL
SELECT XXHASH64(company_key, equipment_code), company_key, equipment_code
FROM equipment WHERE COALESCE(equipment_code, '') <> ''
""")

spark.sql(f"""
CREATE OR REPLACE TABLE {q(GOLD_SCHEMA)}.dim_date
USING DELTA
AS
WITH dates AS (
    SELECT EXPLODE(SEQUENCE(DATE'2015-01-01', ADD_MONTHS(CURRENT_DATE(), 24), INTERVAL 1 DAY)) AS calendar_date
)
SELECT
    CAST(DATE_FORMAT(calendar_date, 'yyyyMMdd') AS INT) AS date_key,
    calendar_date,
    YEAR(calendar_date) AS calendar_year,
    QUARTER(calendar_date) AS calendar_quarter,
    MONTH(calendar_date) AS calendar_month_number,
    DATE_FORMAT(calendar_date, 'MMMM') AS calendar_month_name,
    WEEKOFYEAR(calendar_date) AS calendar_week_number,
    DAY(calendar_date) AS calendar_day_number,
    DATE_FORMAT(calendar_date, 'EEEE') AS calendar_day_name,
    CASE WHEN DAYOFWEEK(calendar_date) IN (1, 7) THEN TRUE ELSE FALSE END AS weekend_flag
FROM dates
""")

StatementMeta(, dffa1368-8c23-41d3-af84-999f7974401e, 21, Finished, Available, Finished, False)

DataFrame[]

## Service-call and appointment facts

In [20]:
spark.sql(f"""
CREATE OR REPLACE TABLE {q(GOLD_SCHEMA)}.fact_service_call
USING DELTA
AS
SELECT
    XXHASH64(s.company_key, s.work_number) AS service_call_key,
    s.company_key,
    s.work_number,
    COALESCE(c.customer_key, 0) AS customer_key,
    COALESCE(l.location_key, 0) AS location_key,
    COALESCE(t.technician_key, 0) AS technician_key,
    COALESCE(k.contract_key, 0) AS contract_key,
    TRY_CAST(DATE_FORMAT(s.call_date, 'yyyyMMdd') AS INT) AS call_date_key,
    TRY_CAST(DATE_FORMAT(s.completion_date, 'yyyyMMdd') AS INT) AS completion_date_key,
    s.status_code,
    s.type_code,
    s.status,
    s.contract_flag,
    h.actual_hours,
    h.budget_hours,
    CASE WHEN s.completion_date IS NOT NULL
         THEN DATEDIFF(s.completion_date, s.call_date) END AS days_to_complete,
    CURRENT_TIMESTAMP() AS _processed_at
FROM {q(SILVER_SCHEMA)}.service_call s
LEFT JOIN {q(SILVER_SCHEMA)}.service_call_hours h
  ON s.company_key = h.company_key AND s.work_number = h.work_number
LEFT JOIN {q(GOLD_SCHEMA)}.dim_customer c
  ON s.company_key = c.company_key AND s.customer_code = c.customer_code
LEFT JOIN {q(GOLD_SCHEMA)}.dim_location l
  ON s.company_key = l.company_key AND s.customer_code = l.customer_code
 AND s.location_code = l.location_code
LEFT JOIN {q(GOLD_SCHEMA)}.dim_technician t
  ON s.company_key = t.company_key
 AND COALESCE(NULLIF(s.technician_assigned, ''), s.technician_primary) = t.technician_username
LEFT JOIN {q(GOLD_SCHEMA)}.dim_contract k
  ON s.company_key = k.company_key
 AND s.customer_code = k.customer_code
 AND s.location_code = k.location_code
 AND s.contract_code = k.contract_number
 AND s.contract_sequence <=> k.contract_sequence
""")

spark.sql(f"""
CREATE OR REPLACE TABLE {q(GOLD_SCHEMA)}.fact_service_appointment
USING DELTA
AS
SELECT
    XXHASH64(a.company_key, a.work_number, a.appointment_number) AS appointment_key,
    XXHASH64(a.company_key, a.work_number) AS service_call_key,
    a.company_key,
    a.work_number,
    a.appointment_number,
    COALESCE(t.technician_key, 0) AS technician_key,
    TRY_CAST(DATE_FORMAT(a.scheduled_date, 'yyyyMMdd') AS INT) AS scheduled_date_key,
    TRY_CAST(DATE_FORMAT(a.completion_date, 'yyyyMMdd') AS INT) AS completion_date_key,
    a.appointment_status,
    a.cost_code,
    a.estimated_hours,
    a.actual_hours,
    CURRENT_TIMESTAMP() AS _processed_at
FROM {q(SILVER_SCHEMA)}.service_appointment a
LEFT JOIN {q(GOLD_SCHEMA)}.dim_technician t
  ON a.company_key = t.company_key AND a.technician_username = t.technician_username
""")

StatementMeta(, dffa1368-8c23-41d3-af84-999f7974401e, 22, Finished, Available, Finished, False)

DataFrame[]

## Cost, labor, contract schedule, and revenue facts

In [21]:
spark.sql(f"""
CREATE OR REPLACE TABLE {q(GOLD_SCHEMA)}.fact_service_cost
USING DELTA
AS
SELECT
    XXHASH64(s.company_key, s.source_record_area, CAST(s.source_row_id AS STRING)) AS service_cost_key,
    XXHASH64(s.company_key, s.work_number) AS service_call_key,
    COALESCE(c.customer_key, 0) AS customer_key,
    COALESCE(l.location_key, 0) AS location_key,
    COALESCE(t.technician_key, 0) AS technician_key,
    COALESCE(e.employee_key, 0) AS employee_key,
    COALESCE(k.contract_key, 0) AS contract_key,
    COALESCE(eq.equipment_key, 0) AS equipment_key,
    s.company_key, s.source_record_area, s.source_row_id,
    s.work_number, s.invoice_number, s.journal_number,
    s.vendor_code, s.vendor_name, s.cost_code, s.cost_label,
    s.cost_description, s.document_code,
    TRY_CAST(DATE_FORMAT(s.document_date, 'yyyyMMdd') AS INT) AS document_date_key,
    s.document_amount, s.billing_amount, s.extended_cost,
    s.ws_extended_cost, s.committed_cost, s.transaction_quantity,
    s.actual_hours, s.billed_hours, s.transaction_source,
    s.transaction_description,
    CURRENT_TIMESTAMP() AS _processed_at
FROM {q(SILVER_SCHEMA)}.service_cost s
LEFT JOIN {q(GOLD_SCHEMA)}.dim_customer c
  ON s.company_key = c.company_key AND s.customer_code = c.customer_code
LEFT JOIN {q(GOLD_SCHEMA)}.dim_location l
  ON s.company_key = l.company_key AND s.customer_code = l.customer_code
 AND s.location_code = l.location_code
LEFT JOIN {q(GOLD_SCHEMA)}.dim_technician t
  ON s.company_key = t.company_key AND s.technician_code = t.technician_username
LEFT JOIN {q(GOLD_SCHEMA)}.dim_employee e
  ON s.company_key = e.company_key
 AND TRY_CAST(s.employee_code_text AS BIGINT) = e.employee_code
LEFT JOIN {q(GOLD_SCHEMA)}.dim_contract k
  ON s.company_key = k.company_key
 AND s.customer_code = k.customer_code AND s.location_code = k.location_code
 AND s.contract_number = k.contract_number AND s.contract_sequence <=> k.contract_sequence
LEFT JOIN {q(GOLD_SCHEMA)}.dim_equipment eq
  ON s.company_key = eq.company_key AND s.equipment_code = eq.equipment_code
""")

spark.sql(f"""
CREATE OR REPLACE TABLE {q(GOLD_SCHEMA)}.fact_service_labor
USING DELTA
AS
SELECT
    XXHASH64(l.company_key, l.work_number, CAST(l.employee_code AS STRING), l.employee_name) AS service_labor_key,
    XXHASH64(l.company_key, l.work_number) AS service_call_key,
    l.company_key, l.work_number,
    COALESCE(e.employee_key, 0) AS employee_key,
    COALESCE(c.customer_key, 0) AS customer_key,
    COALESCE(loc.location_key, 0) AS location_key,
    l.division_code, l.actual_hours,
    CURRENT_TIMESTAMP() AS _processed_at
FROM {q(SILVER_SCHEMA)}.service_labor_actual l
LEFT JOIN {q(GOLD_SCHEMA)}.dim_employee e
  ON l.company_key = e.company_key AND l.employee_code = e.employee_code
LEFT JOIN {q(GOLD_SCHEMA)}.dim_customer c
  ON l.company_key = c.company_key AND l.customer_code = c.customer_code
LEFT JOIN {q(GOLD_SCHEMA)}.dim_location loc
  ON l.company_key = loc.company_key AND l.customer_code = loc.customer_code
 AND l.location_code = loc.location_code
""")

spark.sql(f"""
CREATE OR REPLACE TABLE {q(GOLD_SCHEMA)}.fact_contract_schedule
USING DELTA
AS
SELECT
    XXHASH64(
      s.company_key,
      s.customer_code,
      s.location_code,
      s.contract_number,
      s.contract_sequence,
      CAST(s.scheduled_date AS STRING),
      COALESCE(s.work_number, ''),
      COALESCE(s.equipment_code, ''),
      COALESCE(s.task_code, '')
    ) AS contract_schedule_key,
    s.company_key,
    COALESCE(c.contract_key, 0) AS contract_key,
    COALESCE(e.equipment_key, 0) AS equipment_key,
    TRY_CAST(DATE_FORMAT(s.scheduled_date, 'yyyyMMdd') AS INT) AS scheduled_date_key,
    s.work_number, s.task_code, s.estimated_hours,
    CURRENT_TIMESTAMP() AS _processed_at
FROM {q(SILVER_SCHEMA)}.contract_schedule s
LEFT JOIN {q(GOLD_SCHEMA)}.dim_contract c
  ON s.company_key = c.company_key
 AND s.customer_code = c.customer_code
 AND s.location_code = c.location_code
 AND s.contract_number = c.contract_number
 AND s.contract_sequence <=> c.contract_sequence
LEFT JOIN {q(GOLD_SCHEMA)}.dim_equipment e
  ON s.company_key = e.company_key AND s.equipment_code = e.equipment_code
""")

spark.sql(f"""
CREATE OR REPLACE TABLE {q(GOLD_SCHEMA)}.fact_contract_revenue
USING DELTA
AS
SELECT
    XXHASH64(
      r.company_key,
      r.customer_code,
      r.location_code,
      r.contract_number,
      r.contract_sequence,
      CAST(r.billed_year AS STRING),
      CAST(r.billed_period AS STRING),
      CAST(r.bill_date AS STRING),
      CAST(r.posted_date AS STRING),
      CAST(r.document_amount AS STRING)
    ) AS contract_revenue_key,
    r.company_key,
    COALESCE(c.contract_key, 0) AS contract_key,
    TRY_CAST(DATE_FORMAT(r.bill_date, 'yyyyMMdd') AS INT) AS bill_date_key,
    TRY_CAST(DATE_FORMAT(r.posted_date, 'yyyyMMdd') AS INT) AS posted_date_key,
    r.billed_year, r.billed_period, r.document_amount,
    CURRENT_TIMESTAMP() AS _processed_at
FROM {q(SILVER_SCHEMA)}.contract_revenue_schedule r
LEFT JOIN {q(GOLD_SCHEMA)}.dim_contract c
  ON r.company_key = c.company_key
 AND r.customer_code = c.customer_code
 AND r.location_code = c.location_code
 AND r.contract_number = c.contract_number
 AND r.contract_sequence <=> c.contract_sequence
""")

StatementMeta(, dffa1368-8c23-41d3-af84-999f7974401e, 23, Finished, Available, Finished, False)

DataFrame[]

## Review

In [22]:
for name in ["dim_customer", "dim_location", "dim_employee", "dim_technician",
             "dim_contract", "dim_equipment", "dim_date", "fact_service_call",
             "fact_service_appointment", "fact_service_cost", "fact_service_labor",
             "fact_contract_schedule", "fact_contract_revenue"]:
    print(f"{GOLD_SCHEMA}.{name}: {spark.table(f'{GOLD_SCHEMA}.{name}').count():,} rows")

spark.sql("""
SELECT
    COUNT(*) AS row_count,
    COUNT(DISTINCT contract_key) AS distinct_key_count
FROM gold.dim_contract
""").show()

StatementMeta(, dffa1368-8c23-41d3-af84-999f7974401e, 24, Finished, Available, Finished, False)

gold.dim_customer: 1,010 rows
gold.dim_location: 3,406 rows
gold.dim_employee: 114 rows
gold.dim_technician: 145 rows
gold.dim_contract: 1,994 rows
gold.dim_equipment: 3,196 rows
gold.dim_date: 5,008 rows
gold.fact_service_call: 23,396 rows
gold.fact_service_appointment: 48,518 rows
gold.fact_service_cost: 357 rows
gold.fact_service_labor: 46,441 rows
gold.fact_contract_schedule: 2,195,391 rows
gold.fact_contract_revenue: 12,157 rows
+---------+------------------+
|row_count|distinct_key_count|
+---------+------------------+
|     1994|              1994|
+---------+------------------+

